In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils import data
import numpy as np
import torchvision
from  numpy import exp,absolute
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import time
import os
import copy
import math
from sklearn import svm
import sklearn.model_selection as model_selection
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,f1_score,precision_score ,recall_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.metrics.pairwise import euclidean_distances
from PIL import Image
import cv2
from skimage.feature import local_binary_pattern
import random
import glob
import pickle

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [3]:
#hyper params
lr = 1e-4
bs = 5
val_split = 0.85
num_epoch = 10
num_classes = 2
labels = ['nom', 'attack']  # class names

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

data_transforms = transforms.Compose([
    transforms.Resize([224, 224]),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
lbp_transforms = transforms.Compose([
    transforms.Resize([224, 224]),
    transforms.Grayscale(num_output_channels=1),  # Convert images to grayscale
    transforms.ToTensor(),
    transforms.Normalize([0.485], [0.229])  # Normalization for grayscale images
])
gray_transform = transforms.Grayscale(num_output_channels=1)

In [4]:
class LBPNet(nn.Module):
    def __init__(self, num_classes):
        super(LBPNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 56 * 56, 512)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)
        return x

In [5]:
def get_model1():
    densenet = torchvision.models.densenet161(pretrained=True)
    # Freeze all the layers except the last few layers
    for param in densenet.parameters():
       param.requires_grad = False
    #densenet.classifier = nn.Sequential(nn.Linear(2208, 512), nn.ReLU(), nn.Linear(512, num_classes), nn.Sigmoid())
    densenet.classifier = nn.Sequential(nn.Linear(2208, num_classes),nn.Sigmoid())
    densenet = densenet.to(device)
    return densenet

def get_model3():
    resnet = torchvision.models.resnet50(pretrained=True)
    # Freeze all the layers except the last few layers
    for param in resnet.parameters():
       param.requires_grad = False
    resnet.fc = nn.Linear(2048,num_classes)
    resnet = resnet.to(device)
    return resnet

def get_LBP():
    lbp_net = LBPNet(num_classes)  # Create an instance of LBPNet
    # Move model to the device
    lbp_net = lbp_net.to(device)
    return lbp_net

In [6]:
    model1 = get_model1()
    model1.load_state_dict(torch.load('/content/drive/MyDrive/DataSets/Save_Models/model1.pth'))
    model1.eval()

    model3 = get_model3()
    model3.load_state_dict(torch.load('/content/drive/MyDrive/DataSets/Save_Models/model3.pth'))
    model3.eval()

    model9 = get_LBP()
    model9.load_state_dict(torch.load('/content/drive/MyDrive/DataSets/Save_Models/model9.pth'))
    model9.eval()

    model_path = '/content/drive/MyDrive/DataSets/Save_Models/svm.pk1'
    with open(model_path, 'rb') as file:
      clf = pickle.load(file)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet161_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet161_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/densenet161-8d451a50.pth" to /root/.cache/torch/hub/checkpoints/densenet161-8d451a50.pth
100%|██████████| 110M/110M [00:01<00:00, 77.5MB/s]
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may b

In [7]:
#Listing Trained Models
def get_models(m1,m2,m3):
    return [m1,m2,m3]

In [8]:
def PreProcess_img(input):
  img = cv2.imread(input)
  img = cv2.resize(img, (224, 224))
  img = transforms.ToTensor()(img)
  img = transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])(img)

  # Add batch dimension
  img = img.unsqueeze(0)
  img = img.to(device) #send tensor to GPU
  return img

In [9]:
def get_weighted_score_img(models,inputs):
  num_models = len(models)
  X = np.empty((0,num_models*num_classes))
  Y = np.empty((0),dtype=int)
  inputs = inputs.to(device)
  predictions = set()
  with torch.set_grad_enabled(False):
      x = models[0](inputs)
      _, preds = torch.max(x, 1)
      predictions.add(preds)
      for i in range(1,num_models):
          if models[i] is model9:
            x1 = models[i](gray_transform(inputs))
          else:
            x1 = models[i](inputs)
            _, preds = torch.max(x1, 1)
          predictions.add(preds)
          x = torch.cat((x,x1),dim=1)
      if len(predictions) > 1:
          X = np.append(X,x.cpu().numpy()*3,axis=0)
      else:
          X = np.append(X,x.cpu().numpy(),axis=0)
      #Y = np.append(Y,labels.cpu().numpy(),axis=0)
  return X

In [10]:
def Nom_Score(model, image):
    model.eval()
    #cs = 0.0
    # Make a prediction
    with torch.no_grad():
        output = model(image)

    # Calculate the softmax of the output
    probabilities = F.softmax(output, dim=1)
    # Get the highest probability and its index
    confidence_scores, predicted_classes = torch.max(probabilities, dim=1)
    # Get the predicted class index
    class_index = predicted_classes.item()
    cs = confidence_scores.item()
    if class_index != 0:
      return (1 - cs), class_index
    else:
      return cs,class_index

In [11]:
def normalisation(score):
  max = 1.0081503554335423
  min = -1370.2244674450371
  return (score - min)/(max - min)

In [12]:
originals = get_models(model1,model3,model9)
modelsvm = get_models(model1,clf,model9)

In [13]:
def Return_Scores_all(originals, models, image):
    Score = []
    Pred = []
    originals = get_models(model1,model3,model9)
    for model in models:
        if model is model9:    #LBP
            gray_transform = transforms.Grayscale(num_output_channels=1)
            img_gray = gray_transform(image)
            # Make a prediction
            c,p = Nom_Score(model, img_gray)
            #print(p, " Three")
        elif model is clf:      #SVM
            img = get_weighted_score_img(originals,image)
            conf = model.decision_function(img)
            conf = normalisation(conf)
            my_list = conf.tolist()
            c = my_list[0]
            pred = model.predict(img)
            my_list1 = pred.tolist()
            p = my_list1[0]
            #print(p, " Two")
        else:
            c,p = Nom_Score(model, image)
            #print(p, "One")
        Score.append(c)
        Pred.append(p)
    #print(Score," ", pred)
    return Score,Pred

In [14]:
img_path = "/content/drive/MyDrive/DataSets/Palmvein_h/Registration/001-M/enrol/001_L_1_1.png"
img = PreProcess_img(img_path)
score,pred = Return_Scores_all(originals,modelsvm,img)
print(score," ", pred)

[0.5051828026771545, 0.9999883931494196, 0.703952968120575]   [0, 1, 0]


In [15]:
img_path = "/content/drive/MyDrive/DataSets/Palmvein_h/Registration/001-M/enrol/001_R_1_1.png"
img = PreProcess_img(img_path)
score,pred = Return_Scores_all(originals,modelsvm,img)
print(score," ", pred)

[0.4730682969093323, 0.9999912753402197, 0.30654239654541016]   [1, 1, 1]


In [16]:
def Real_Count(list):
  r = 0
  n = len(list)
  for i in range(n):
    if(list[i] == 0):
      r = r + 1
  return r

In [23]:
# Normal Operation Mode
print("NORMAL OPERATION MODE")
#FRR - False Rejection Rate (Genuine inputs rejected as False)
nom_path = "/content/drive/MyDrive/DataSets/Palmvein_h/test/enrol/nom"
image_files = glob.glob(nom_path + '/*.png')  # Modify the file extension as per your image format
total1 = len(image_files)
print("Total Genuine Inputs (Enrolment): ",total1)
i=0
frr = 0  #valid input reject count

for image_file in image_files:
    image = PreProcess_img(image_file)
    confidence,prediction = Nom_Score(model3, image)
    #r = Real_Count(prediction)
    if(prediction != 0): #if(r < 2):
      frr += 1
print("Genuine Inputs Rejected:",frr)
print("FRR % is :",((frr * 100)/total1))


#FAR - False Acceptance Rate (Zero-Effort Attack/ Real Images used by Attacker to Attack)
nom_path = "/content/drive/MyDrive/DataSets/Palmvein_h/test/probe/nom"
image_files = glob.glob(nom_path + '/*.png')  # Modify the file extension as per your image format
total2 = len(image_files)
print("Total Real Probe Inputs by Attacker(Nom): ",total2)
i=0
FAR = []
far = 0  #invalid input accept count

for image_file in image_files:
    image = PreProcess_img(image_file)
    confidence,prediction = Nom_Score(model3, image)
    #r = Real_Count(prediction)
    if(prediction == 0): #if(r >= 2):
      far += 1
print("Real Probes Accepted:",far)
print("FAR % is :",((far * 100)/total2))

print(" ")

#EER Calculation - Equal Error Rate (Average of FRR and FAR)
eer = ((frr/total1) + (far/total2))/2
print("EER % is :",eer*100)

#Spoofing Attack
# SFAR Calculation - Spoofing FAR (Incorrect or Spoof Images Accepted Rate)
print("\nSPOOFING ATTACK MODE")
spoof_path = "/content/drive/MyDrive/DataSets/Palmvein_h/test/probe/attack"
image_files = glob.glob(spoof_path + '/*.png')  # Modify the file extension as per your image format
total3 = len(image_files)
print("Total Spoof Inputs: ",total3)
i=0
nom_count = 0  #false image accepted as nom count

for image_file in image_files:
    image = PreProcess_img(image_file)
    confidence,prediction = Nom_Score(model3, image)
    #r = Real_Count(prediction)
    if(prediction == 0): #if(r >= 2):
      nom_count += 1
print("Spoof Images Accepted:",nom_count)
print("SFAR % is :",((nom_count * 100)/total3))

NORMAL OPERATION MODE
Total Genuine Inputs (Enrolment):  180
Genuine Inputs Rejected: 140
FRR % is : 77.77777777777777
Total Real Probe Inputs by Attacker(Nom):  720
Real Probes Accepted: 147
FAR % is : 20.416666666666668
 
EER % is : 49.09722222222223

SPOOFING ATTACK MODE
Total Spoof Inputs:  720
Spoof Images Accepted: 500
SFAR % is : 69.44444444444444
